In [1]:
import operator
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

load_dotenv()

# Initialize Groq LLM
model = ChatGroq(model_name="llama-3.3-70b-versatile")


# Schema for structured output
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)


structured_model = model.with_structured_output(EvaluationSchema)


# State definition with Annotated list reducer for parallel score aggregation
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float


# Parallel Node 1: Evaluate Language Quality
def evaluate_language(state: UPSCState) -> dict:
    prompt = f"Evaluate the language quality of the following essay, provide feedback, and assign a score out of 10:\n\n{state['essay']}"
    output = structured_model.invoke(prompt)
    return {
        "language_feedback": output.feedback,
        "individual_scores": [output.score],
    }


# Parallel Node 2: Evaluate Depth of Analysis
def evaluate_analysis(state: UPSCState) -> dict:
    prompt = f"Evaluate the depth of analysis of the following essay, provide feedback, and assign a score out of 10:\n\n{state['essay']}"
    output = structured_model.invoke(prompt)
    return {
        "analysis_feedback": output.feedback,
        "individual_scores": [output.score],
    }


# Parallel Node 3: Evaluate Clarity of Thought
def evaluate_thought(state: UPSCState) -> dict:
    prompt = f"Evaluate the clarity of thought of the following essay, provide feedback, and assign a score out of 10:\n\n{state['essay']}"
    output = structured_model.invoke(prompt)
    return {
        "clarity_feedback": output.feedback,
        "individual_scores": [output.score],
    }


# Final Node: Summarize Feedback & Compute Average Score
def final_evaluation(state: UPSCState) -> dict:
    prompt = (
        f"Based on the following feedback, create a summarized feedback report:\n\n"
        f"Language Feedback: {state['language_feedback']}\n"
        f"Depth of Analysis Feedback: {state['analysis_feedback']}\n"
        f"Clarity of Thought Feedback: {state['clarity_feedback']}"
    )
    overall_feedback = model.invoke(prompt).content

    scores = state["individual_scores"]
    avg_score = round(sum(scores) / len(scores), 2) if scores else 0.0

    return {"overall_feedback": overall_feedback, "avg_score": avg_score}


# Build Graph Workflow
graph = StateGraph(UPSCState)

# Add Nodes
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# Parallel Fan-Out Edges from START
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

# Fan-In Edges into Final Evaluation
graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")

# Complete Graph
graph.add_edge("final_evaluation", END)

workflow = graph.compile()


# Execution
essay_input = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

initial_state = {"essay": essay_input}
final_state = workflow.invoke(initial_state)

# Display Results
print("=== INDIVIDUAL SCORES ===")
print(final_state["individual_scores"])

print("\n=== AVERAGE SCORE ===")
print(final_state["avg_score"])

print("\n=== OVERALL SUMMARY FEEDBACK ===")
print(final_state["overall_feedback"])

=== INDIVIDUAL SCORES ===
[4, 4, 6]

=== AVERAGE SCORE ===
4.67

=== OVERALL SUMMARY FEEDBACK ===
**Summarized Feedback Report**

The essay demonstrates a clear understanding of the impact of AI on India, highlighting both benefits and challenges. However, there are areas for improvement in language quality, depth of analysis, and clarity of thought.

**Key Strengths:**

* Demonstrates a clear understanding of the topic
* Attempts to offer solutions and emphasizes collective effort

**Key Areas for Improvement:**

* Language quality:
	+ Use more complex sentence structures
	+ Improve vocabulary
	+ Enhance grammar
* Depth of analysis:
	+ Provide concrete examples and supporting evidence
	+ Incorporate statistical data and expert opinions
* Clarity of thought:
	+ Develop a more sophisticated writing style
	+ Use transitional phrases to connect ideas
	+ Provide nuance in analysis and acknowledge complexity of issues

**Recommendations:**

* Focus on developing a more varied and sophistica